In [1]:
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
)

# ============================================================
# CONFIGURATION
# ============================================================
CSV_FILE = "NIFTY 50_Historical_PR_02032026to01092026.csv"
OUTPUT_DIR = "nifty50_analysis_output"
FORECAST_DAYS = 5
BACKTEST_DAYS = 20
RANDOM_STATE = 42

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ============================================================
# 1. LOAD + CLEAN HISTORICAL DATA
# ============================================================
print("=" * 80)
print("NIFTY 50 - COMPLETE ANALYTICS PROGRAM")
print("=" * 80)

if not os.path.exists(CSV_FILE):
    raise FileNotFoundError(f"CSV file not found: {CSV_FILE}")

df = pd.read_csv(CSV_FILE)

df.columns = [c.strip() for c in df.columns]

required = ["Date", "Open", "High", "Low", "Close"]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}. Found: {list(df.columns)}")

# Date conversion and sorting
# The supplied CSV is newest-first, so sorting is essential for time-series work.
df["Date"] = pd.to_datetime(df["Date"], dayfirst=True, errors="coerce")
for col in ["Open", "High", "Low", "Close"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.dropna(subset=required).copy()
df = df.drop_duplicates(subset=["Date"]).sort_values("Date").reset_index(drop=True)

# ============================================================
# 2. FEATURE ENGINEERING
# ============================================================
df["Daily_Return"] = df["Close"].pct_change()
df["Daily_Return_Pct"] = df["Daily_Return"] * 100

df["Price_Change"] = df["Close"].diff()
df["Intraday_Range"] = df["High"] - df["Low"]
df["Intraday_Range_Pct"] = (df["High"] - df["Low"]) / df["Close"] * 100

df["MA_5"] = df["Close"].rolling(5).mean()
df["MA_10"] = df["Close"].rolling(10).mean()
df["MA_20"] = df["Close"].rolling(20).mean()

df["Volatility_10D"] = df["Daily_Return"].rolling(10).std() * np.sqrt(252) * 100

df["Momentum_5D"] = df["Close"] / df["Close"].shift(5) - 1

df["Lag_1"] = df["Close"].shift(1)
df["Lag_2"] = df["Close"].shift(2)
df["Lag_3"] = df["Close"].shift(3)
df["Lag_5"] = df["Close"].shift(5)
df["Lag_10"] = df["Close"].shift(10)

# Classification target: 1 = next trading day closes higher, 0 = lower/equal
# Current-row target is based on next day's close, so it can be used for supervised learning.
df["Target_Next_Day_Up"] = (df["Close"].shift(-1) > df["Close"]).astype(int)

# ============================================================
# 3. HISTORICAL DATA OUTPUT
# ============================================================
print("\n[1] HISTORICAL DATA")
print("Date range:", df["Date"].min().date(), "to", df["Date"].max().date())
print("Trading days:", len(df))
print("\nLatest 10 records:")
print(df[["Date", "Open", "High", "Low", "Close"]].tail(10).to_string(index=False))

df.to_csv(os.path.join(OUTPUT_DIR, "historical_and_engineered_data.csv"), index=False)

# ============================================================
# 4. DESCRIPTIVE ANALYTICS
# ============================================================
print("\n[2] DESCRIPTIVE ANALYTICS")

summary = df[["Open", "High", "Low", "Close", "Daily_Return_Pct", "Intraday_Range_Pct"]].describe().T
summary["median"] = df[["Open", "High", "Low", "Close", "Daily_Return_Pct", "Intraday_Range_Pct"]].median()
summary = summary[["count", "mean", "median", "std", "min", "25%", "50%", "75%", "max"]]
print(summary.round(3).to_string())

first_close = df["Close"].iloc[0]
last_close = df["Close"].iloc[-1]
overall_return = (last_close / first_close - 1) * 100
avg_daily_return = df["Daily_Return"].mean() * 100
annualized_volatility = df["Daily_Return"].std() * np.sqrt(252) * 100
up_days = int((df["Daily_Return"] > 0).sum())
down_days = int((df["Daily_Return"] < 0).sum())

print(f"\nOverall return: {overall_return:.2f}%")
print(f"Average daily return: {avg_daily_return:.3f}%")
print(f"Annualized volatility: {annualized_volatility:.2f}%")
print(f"Up days: {up_days} | Down days: {down_days}")

summary.to_csv(os.path.join(OUTPUT_DIR, "descriptive_statistics.csv"))

# ============================================================
# 5. DIAGNOSTIC ANALYTICS
# ============================================================
print("\n[3] DIAGNOSTIC ANALYTICS")

largest_gain_idx = df["Daily_Return"].idxmax()
largest_loss_idx = df["Daily_Return"].idxmin()

print("Largest gain:")
print(df.loc[largest_gain_idx, ["Date", "Close", "Daily_Return_Pct"]].to_string())
print("\nLargest loss:")
print(df.loc[largest_loss_idx, ["Date", "Close", "Daily_Return_Pct"]].to_string())

# Drawdown analysis
running_max = df["Close"].cummax()
df["Drawdown_Pct"] = (df["Close"] / running_max - 1) * 100
max_drawdown = df["Drawdown_Pct"].min()
max_drawdown_date = df.loc[df["Drawdown_Pct"].idxmin(), "Date"].date()
print(f"\nMaximum drawdown: {max_drawdown:.2f}% on {max_drawdown_date}")

# Trend position
latest = df.iloc[-1]
trend_comment = "Bullish" if latest["Close"] > latest["MA_20"] else "Bearish/Below 20-day MA"
print(f"Latest close: {latest['Close']:.2f}")
print(f"Latest 20-day MA: {latest['MA_20']:.2f}")
print(f"Current trend signal: {trend_comment}")

diagnostic = df[[
    "Date", "Close", "Daily_Return_Pct", "Intraday_Range_Pct",
    "MA_5", "MA_10", "MA_20", "Volatility_10D", "Drawdown_Pct"
]].copy()
diagnostic.to_csv(os.path.join(OUTPUT_DIR, "diagnostic_metrics.csv"), index=False)

# ============================================================
# 6. CORRELATION ANALYSIS
# ============================================================
print("\n[4] CORRELATION ANALYSIS")

corr_cols = ["Open", "High", "Low", "Close", "Daily_Return", "Intraday_Range"]
corr = df[corr_cols].corr()
print(corr.round(3).to_string())
corr.to_csv(os.path.join(OUTPUT_DIR, "correlation_matrix.csv"))

plt.figure(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, square=True)
plt.title("NIFTY 50 Correlation Matrix")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "01_correlation_heatmap.png"), dpi=160)
plt.close()

# ============================================================
# 7. LINEAR REGRESSION - NEXT DAY CLOSE
# ============================================================
print("\n[5] LINEAR REGRESSION")

reg_features = ["Open", "High", "Low", "MA_5", "MA_10", "MA_20", "Daily_Return", "Intraday_Range"]
# Predict the NEXT trading day's closing value from information available today.
df["Next_Day_Close"] = df["Close"].shift(-1)
reg_data = df.dropna(subset=reg_features + ["Next_Day_Close"]).copy()

# Honest time-ordered evaluation: earliest 80% train, latest 20% test.
split_idx = int(len(reg_data) * 0.80)
train_reg = reg_data.iloc[:split_idx]
test_reg = reg_data.iloc[split_idx:]

X_train = train_reg[reg_features]
y_train = train_reg["Next_Day_Close"]
X_test = test_reg[reg_features]
y_test = test_reg["Next_Day_Close"]

reg_model = LinearRegression()
reg_model.fit(X_train, y_train)
reg_pred = reg_model.predict(X_test)

reg_mae = mean_absolute_error(y_test, reg_pred)
reg_rmse = np.sqrt(mean_squared_error(y_test, reg_pred))
reg_r2 = r2_score(y_test, reg_pred)

print(f"MAE : {reg_mae:.2f}")
print(f"RMSE: {reg_rmse:.2f}")
print(f"R²  : {reg_r2:.4f}")

reg_results = pd.DataFrame({
    "Date": test_reg["Date"].values,
    "Actual_Next_Day_Close": y_test.values,
    "Predicted_Next_Day_Close": reg_pred,
})
reg_results["Absolute_Error"] = abs(reg_results["Actual_Next_Day_Close"] - reg_results["Predicted_Next_Day_Close"])
reg_results.to_csv(os.path.join(OUTPUT_DIR, "linear_regression_results.csv"), index=False)

print("\nRegression coefficients:")
coef_table = pd.DataFrame({"Feature": reg_features, "Coefficient": reg_model.coef_})
print(coef_table.sort_values("Coefficient", ascending=False).round(4).to_string(index=False))

plt.figure(figsize=(11, 5))
plt.plot(test_reg["Date"], y_test, label="Actual Next-Day Close")
plt.plot(test_reg["Date"], reg_pred, label="Predicted Next-Day Close")
plt.title("Linear Regression: Actual vs Predicted Next-Day Close")
plt.xlabel("Date")
plt.ylabel("NIFTY 50 Close")
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "02_linear_regression_actual_vs_predicted.png"), dpi=160)
plt.close()

# Regression scatter
plt.figure(figsize=(7, 6))
plt.scatter(y_test, reg_pred, alpha=0.75)
min_v = min(y_test.min(), reg_pred.min())
max_v = max(y_test.max(), reg_pred.max())
plt.plot([min_v, max_v], [min_v, max_v], linestyle="--")
plt.title("Linear Regression: Actual vs Predicted")
plt.xlabel("Actual Next-Day Close")
plt.ylabel("Predicted Next-Day Close")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "03_linear_regression_scatter.png"), dpi=160)
plt.close()

# ============================================================
# 8. CLASSIFICATION - NEXT DAY UP / DOWN
# ============================================================
print("\n[6] CLASSIFICATION")

class_features = ["Daily_Return", "MA_5", "MA_10", "MA_20", "Volatility_10D", "Momentum_5D", "Intraday_Range_Pct"]
class_data = df.dropna(subset=class_features + ["Target_Next_Day_Up"]).copy()

split_cls = int(len(class_data) * 0.80)
train_cls = class_data.iloc[:split_cls]
test_cls = class_data.iloc[split_cls:]

Xc_train = train_cls[class_features]
yc_train = train_cls["Target_Next_Day_Up"]
Xc_test = test_cls[class_features]
yc_test = test_cls["Target_Next_Day_Up"]

# Time-ordered evaluation; no shuffling.
clf = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE)),
])
clf.fit(Xc_train, yc_train)
yc_pred = clf.predict(Xc_test)

acc = accuracy_score(yc_test, yc_pred)
prec = precision_score(yc_test, yc_pred, zero_division=0)
rec = recall_score(yc_test, yc_pred, zero_division=0)
f1 = f1_score(yc_test, yc_pred, zero_division=0)
cm = confusion_matrix(yc_test, yc_pred)

print(f"Accuracy : {acc:.4f} ({acc*100:.2f}%)")
print(f"Precision: {prec:.4f}")
print(f"Recall   : {rec:.4f}")
print(f"F1-score : {f1:.4f}")
print("\nClassification report:")
print(classification_report(yc_test, yc_pred, target_names=["DOWN", "UP"], zero_division=0))

class_results = pd.DataFrame({
    "Date": test_cls["Date"].values,
    "Actual": yc_test.values,
    "Predicted": yc_pred,
    "Actual_Label": np.where(yc_test.values == 1, "UP", "DOWN"),
    "Predicted_Label": np.where(yc_pred == 1, "UP", "DOWN"),
})
class_results.to_csv(os.path.join(OUTPUT_DIR, "classification_results.csv"), index=False)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["DOWN", "UP"], yticklabels=["DOWN", "UP"])
plt.title("Classification Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "04_classification_confusion_matrix.png"), dpi=160)
plt.close()

# ============================================================
# 9. FIVE-DAY FORECAST USING AUTOREGRESSIVE LINEAR REGRESSION
# ============================================================
print("\n[7] 5-DAY FORECAST")

# Use lagged closes to forecast the next close recursively.
lag_features = ["Lag_1", "Lag_2", "Lag_3", "Lag_5", "Lag_10"]
lag_data = df.dropna(subset=lag_features + ["Close"]).copy()

# Train on all available historical data because the goal is future forecasting.
X_lag = lag_data[lag_features]
y_lag = lag_data["Close"]

forecast_model = LinearRegression()
forecast_model.fit(X_lag, y_lag)

history = df["Close"].tolist()
future_dates = pd.bdate_range(df["Date"].max() + pd.Timedelta(days=1), periods=FORECAST_DAYS)
forecast_values = []

for _ in range(FORECAST_DAYS):
    # Build lags from the most recent history, recursively using forecasts.
    x_next = np.array([[history[-1], history[-2], history[-3], history[-5], history[-10]]])
    next_pred = float(forecast_model.predict(x_next)[0])
    forecast_values.append(next_pred)
    history.append(next_pred)

forecast_df = pd.DataFrame({
    "Forecast_Date": future_dates,
    "Forecast_Close": forecast_values,
})

print(forecast_df.to_string(index=False, formatters={"Forecast_Close": "{:.2f}".format}))
forecast_df.to_csv(os.path.join(OUTPUT_DIR, "5_day_forecast.csv"), index=False)

# Forecast chart using last 30 actual observations + 5 forecasts
plot_history = df.tail(30)
plt.figure(figsize=(12, 6))
plt.plot(plot_history["Date"], plot_history["Close"], label="Historical Close")
plt.plot(forecast_df["Forecast_Date"], forecast_df["Forecast_Close"], marker="o", linestyle="--", label="5-Day Forecast")
plt.axvline(df["Date"].max(), linestyle=":")
plt.title("NIFTY 50 - 5-Day Forecast")
plt.xlabel("Date")
plt.ylabel("Close")
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "05_five_day_forecast.png"), dpi=160)
plt.close()

# ============================================================
# 10. ADDITIONAL VISUALIZATIONS
# ============================================================
print("\n[8] VISUALIZATIONS")

# Price + moving averages
plt.figure(figsize=(12, 6))
plt.plot(df["Date"], df["Close"], label="Close")
plt.plot(df["Date"], df["MA_5"], label="MA 5")
plt.plot(df["Date"], df["MA_20"], label="MA 20")
plt.title("NIFTY 50 Price and Moving Averages")
plt.xlabel("Date")
plt.ylabel("Index Level")
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "06_price_moving_averages.png"), dpi=160)
plt.close()

# Daily returns
plt.figure(figsize=(12, 5))
plt.plot(df["Date"], df["Daily_Return_Pct"])
plt.axhline(0, linestyle="--")
plt.title("NIFTY 50 Daily Returns")
plt.xlabel("Date")
plt.ylabel("Return (%)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "07_daily_returns.png"), dpi=160)
plt.close()

# Rolling volatility
plt.figure(figsize=(12, 5))
plt.plot(df["Date"], df["Volatility_10D"])
plt.title("10-Day Rolling Annualized Volatility")
plt.xlabel("Date")
plt.ylabel("Volatility (%)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "08_rolling_volatility.png"), dpi=160)
plt.close()

# Drawdown
plt.figure(figsize=(12, 5))
plt.plot(df["Date"], df["Drawdown_Pct"])
plt.axhline(0, linestyle="--")
plt.title("NIFTY 50 Drawdown")
plt.xlabel("Date")
plt.ylabel("Drawdown (%)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "09_drawdown.png"), dpi=160)
plt.close()

# ============================================================
# 11. CONSOLIDATED KPI / REPORT
# ============================================================
metrics = {
    "Data_Start": str(df["Date"].min().date()),
    "Data_End": str(df["Date"].max().date()),
    "Trading_Days": len(df),
    "Starting_Close": float(first_close),
    "Latest_Close": float(last_close),
    "Overall_Return_Pct": float(overall_return),
    "Average_Daily_Return_Pct": float(avg_daily_return),
    "Annualized_Volatility_Pct": float(annualized_volatility),
    "Up_Days": up_days,
    "Down_Days": down_days,
    "Max_Drawdown_Pct": float(max_drawdown),
    "Regression_MAE": float(reg_mae),
    "Regression_RMSE": float(reg_rmse),
    "Regression_R2": float(reg_r2),
    "Classification_Accuracy": float(acc),
    "Classification_Precision": float(prec),
    "Classification_Recall": float(rec),
    "Classification_F1": float(f1),
}

pd.DataFrame([metrics]).to_csv(os.path.join(OUTPUT_DIR, "summary_metrics.csv"), index=False)

print("\n" + "=" * 80)
print("FINAL SUMMARY")
print("=" * 80)
for k, v in metrics.items():
    print(f"{k}: {v}")
print("\nFiles created in:", OUTPUT_DIR)
print("- historical_and_engineered_data.csv")
print("- descriptive_statistics.csv")
print("- diagnostic_metrics.csv")
print("- correlation_matrix.csv")
print("- linear_regression_results.csv")
print("- classification_results.csv")
print("- 5_day_forecast.csv")
print("- summary_metrics.csv")
print("- 9 PNG visualizations")
print("=" * 80)


NIFTY 50 - COMPLETE ANALYTICS PROGRAM

[1] HISTORICAL DATA
Date range: 2026-03-02 to 2026-09-01
Trading days: 124

Latest 10 records:
      Date     Open     High      Low    Close
2026-08-19 24152.05 24172.85 24025.65 24078.30
2026-08-20 24225.45 24265.15 24184.55 24231.85
2026-08-21 24284.05 24284.05 24206.80 24252.00
2026-08-24 24285.05 24313.00 24144.30 24219.05
2026-08-25 24175.75 24334.55 24115.45 24334.55
2026-08-26 24341.95 24378.60 24207.75 24207.75
2026-08-27 24277.60 24297.45 24090.85 24090.85
2026-08-28 24122.60 24188.30 24076.85 24175.65
2026-08-31 24117.55 24128.70 23993.60 24080.40
2026-09-01 24077.55 24143.15 23952.55 24055.80

[2] DESCRIPTIVE ANALYTICS
                    count       mean     median      std        min        25%        50%        75%        max
Open                124.0  23924.787  24057.175  490.830  22383.400  23669.988  24057.175  24237.625  24703.900
High                124.0  24043.541  24135.100  454.165  22714.100  23830.762  24135.100  24318.3